In [1]:
# how to uncover truths that don't matter - fourth section

import time

import numpy as np
import coriolis_module

import matplotlib.pyplot as plt 
import seaborn as sns

import pandas as pd
from sklearn.linear_model import LinearRegression

# loading dataframe with airport information
ldf = pd.read_csv("data/airports.csv").astype(
    {
    "IATA": "category", 
    "AIRPORT": "category", 
    "CITY": "category", 
    "STATE": "category", 
    "COUNTRY": "category", 
    }
)

# loading dataframe with not cancelled flights
fdf = pd.read_csv("data/not_cancelled_flights.csv").astype(
    {
     "FL_DATE": "datetime64[ns]", 
     "AIRLINE": "category", 
     "AIRLINE_DOT": "category", 
     "AIRLINE_CODE": "category", 
     "ORIGIN_CITY": "category", 
     "DEST_CITY": "category", 
     "CANCELLATION_CODE": "category", 
     "DIVERTED": "bool",
     "arr_datetime": "datetime64[ns]",
     "dep_datetime": "datetime64[ns]", 
     "crs_arr_datetime": "datetime64[ns]", 
     "crs_dep_datetime": "datetime64[ns]", 
     "woff_datetime": "datetime64[ns]",
     "won_datetime": "datetime64[ns]", 
     "ORIGIN": "category", 
     "DEST": "category",
    }
)

In [2]:
# Merge fdf with ldf to add geographical data for ORIGIN and DEST
fdf_merged = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="ORIGIN", right_on="IATA", how="left")
fdf_merged = pd.merge(fdf_merged, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="DEST", right_on="IATA", how="left", suffixes=("_ORIGIN", "_DEST"))

# Drop original origin and destination columns
fdf_merged = fdf_merged.drop(["ORIGIN", "DEST"], axis=1)

In [3]:
print("number of rows before cleaning: ", len(fdf_merged))

number of rows before cleaning:  2870784


In [4]:
fdf_merged.dropna(subset=["total_drift_distance", "haversine_distance"], inplace=True)

In [5]:
print("number of rows after cleaning: ", len(fdf_merged))

number of rows after cleaning:  2863927


In [6]:
# division of drift through distance gives percentage
fdf_merged.loc[:, "drift_factor"] = fdf_merged["total_drift_distance"] / fdf_merged["haversine_distance"].replace(0, pd.NA)

In [8]:
m, b = np.polyfit(fdf_merged["haversine_distance"], fdf_merged["drift_factor"],1 )

In [9]:
print("Slope (m): ", m)
print("Intercept (b): ", b)

Slope (m):  0.0001543949697630894
Intercept (b):  -0.02012230911449961


In [10]:
X = fdf_merged["haversine_distance"].values.reshape(-1, 1)
Y = fdf_merged["drift_factor"].values

model = LinearRegression()
model.fit(X, Y)

m = model.coef_[0]
b = model.intercept_

In [11]:
print("Slope (m): ", m)
print("Intercept (b): ", b)

Slope (m):  0.00015439496976308952
Intercept (b):  -0.020122309114499998
